# **VAEP e Synergy VAEP**

## **1. Ambiente**

In [ ]:
import ast
import json
import pickle
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


## **2. Configuração**

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH        = PROJECT_ROOT / 'data'
PROCESSED_PATH   = DATA_PATH / 'processed'
VIZ_PATH         = PROJECT_ROOT / 'viz'
EVENTS_PATH      = DATA_PATH / 'raw' / 'combined' / 'all_events_combined.parquet'
MATCHES_PATH     = DATA_PATH / 'raw' / 'combined' / 'all_matches_combined.parquet'
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
VIZ_PATH.mkdir(parents=True, exist_ok=True)

HORIZON          = 10
RANDOM_STATE     = 42
RECOMPUTE_VAEP   = False
VAEP_CACHE_VERSION = 'spadl_flat_statsbomb_v1'

SB_FIELD_LENGTH  = 120.0
SB_FIELD_WIDTH   = 80.0
SPADL_LENGTH     = 105.0
SPADL_WIDTH      = 68.0
GOAL_WIDTH       = 7.32

XGB_PARAMS = dict(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    enable_categorical=True,
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=25,
    reg_lambda=2.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

PROJECT_ROOT


## **3. Carregamento de Dados**

In [ ]:
EVENT_COLUMNS = [
    'id', 'match_id', 'period', 'minute', 'second', 'timestamp', 'index',
    'type', 'team', 'team_id', 'player', 'player_id', 'position',
    'play_pattern', 'possession', 'possession_team', 'possession_team_id',
    'location_x', 'location_y', 'pass_end_location_x', 'pass_end_location_y',
    'carry_end_location_x', 'carry_end_location_y', 'shot_end_location',
    'goalkeeper_end_location',
    'pass_type', 'pass_height', 'pass_cross', 'pass_outcome', 'pass_body_part',
    'dribble_outcome', 'duel_type', 'duel_outcome', 'interception_outcome',
    'shot_type', 'shot_outcome', 'shot_body_part', 'shot_statsbomb_xg',
    'goalkeeper_type', 'goalkeeper_outcome', 'goalkeeper_body_part',
    'clearance_body_part', 'foul_committed_card',
    'ball_receipt_outcome', 'ball_recovery_recovery_failure', 'out',
    'substitution_replacement', 'substitution_replacement_id', 'tactics',
]

available_event_cols = pq.read_schema(EVENTS_PATH).names
EVENT_COLUMNS = [c for c in EVENT_COLUMNS if c in available_event_cols]

events  = pd.read_parquet(EVENTS_PATH, columns=EVENT_COLUMNS)
matches = pd.read_parquet(MATCHES_PATH)

events = events.sort_values(['match_id', 'period', 'minute', 'second', 'index']).reset_index(drop=True)
events['event_minute'] = events['minute'].astype(float) + events['second'].fillna(0).astype(float) / 60.0

print(f'Eventos: {len(events):,}  |  Partidas: {len(matches):,}')


## **4. Conversão StatsBomb → SPADL-like**

O VAEP base usa ações on-ball em uma linguagem padronizada. Aqui os eventos StatsBomb achatados são convertidos para uma representação SPADL-like compatível com o mapeamento do `socceraction`: eventos espelhados ou contextuais como `Pressure`, `Block`, `Dribbled Past` e `Foul Won` não entram como ações independentes para evitar duplicação e troca artificial de posse.


In [ ]:
SPADL_ACTION_RAW_TYPES = {
    'Pass', 'Carry', 'Dribble', 'Shot', 'Duel', 'Interception',
    'Clearance', 'Goal Keeper', 'Foul Committed', 'Miscontrol', 'Own Goal Against',
}

EXCLUDED_DEFENSIVE_CONTEXT_TYPES = {'Pressure', 'Block', 'Dribbled Past', 'Foul Won'}
SUCCESS_RESULTS = {'Won', 'Success In Play', 'Success Out'}
FAIL_GK = {'Fail', 'Lost In Play', 'Lost Out', 'In Play Danger', 'No Touch'}
SHOT_GOALKEEPER_TYPES = {
    'Shot Saved', 'Shot Saved to Post', 'Shot Saved Off Target', 'Penalty Saved',
    'Penalty Saved to Post', 'Save', 'Saved to Post', 'Smother',
}
CLAIM_GOALKEEPER_TYPES = {'Collected', 'Keeper Sweeper'}
PUNCH_GOALKEEPER_TYPES = {'Punch'}


def _scale_x(s):
    return pd.to_numeric(s, errors='coerce') * SPADL_LENGTH / SB_FIELD_LENGTH


def _scale_y(s):
    return SPADL_WIDTH - (pd.to_numeric(s, errors='coerce') * SPADL_WIDTH / SB_FIELD_WIDTH)


def _list_item(value, pos):
    if isinstance(value, (list, tuple, np.ndarray)) and len(value) > pos:
        return value[pos]
    return np.nan


def _truthy_series(s):
    return s.eq(True) | s.astype('string').str.lower().eq('true')


def _normalize_bodypart(s):
    out = s.astype('string').str.lower().fillna('none')
    out = out.str.replace(' ', '_', regex=False)
    out = out.replace({
        '<na>': 'none',
        'nan': 'none',
        'right_foot': 'right_foot',
        'left_foot': 'left_foot',
        'head': 'head',
        'keeper_arm': 'other',
        'drop_kick': 'right_foot',
        'right_hand': 'other',
        'left_hand': 'other',
        'both_hands': 'other',
        'chest': 'other',
        'no_touch': 'none',
    })
    return out.where(out.isin(['right_foot', 'left_foot', 'head', 'other', 'none']), 'other')


def add_match_team_ids(matches_df, events_df):
    team_lk = (
        events_df.dropna(subset=['match_id', 'team_id', 'team'])
        .assign(team_id=lambda d: d['team_id'].astype(int))
        .drop_duplicates(['match_id', 'team'])
        .set_index(['match_id', 'team'])['team_id']
        .to_dict()
    )
    team_ids_by_match = (
        events_df.dropna(subset=['match_id', 'team_id'])
        .assign(team_id=lambda d: d['team_id'].astype(int))
        .groupby('match_id')['team_id']
        .agg(lambda x: tuple(pd.unique(x)))
        .to_dict()
    )
    mt = matches_df.copy()
    mt['home_team_id'] = [team_lk.get((r.match_id, r.home_team)) for r in mt.itertuples(index=False)]
    mt['away_team_id'] = [team_lk.get((r.match_id, r.away_team)) for r in mt.itertuples(index=False)]
    for idx, row in mt.loc[mt['home_team_id'].isna() | mt['away_team_id'].isna()].iterrows():
        ids = team_ids_by_match.get(row['match_id'], ())
        if len(ids) == 2:
            if pd.isna(row['home_team_id']):
                mt.at[idx, 'home_team_id'] = ids[0]
            if pd.isna(row['away_team_id']):
                mt.at[idx, 'away_team_id'] = ids[1] if ids[0] == mt.at[idx, 'home_team_id'] else ids[0]
    mt['home_team_id'] = mt['home_team_id'].astype('Int64')
    mt['away_team_id'] = mt['away_team_id'].astype('Int64')
    return mt


def build_opponent_lookup(match_meta):
    lookup = {}
    for row in match_meta.dropna(subset=['home_team_id', 'away_team_id']).itertuples(index=False):
        h, a = int(row.home_team_id), int(row.away_team_id)
        lookup[(row.match_id, h)] = a
        lookup[(row.match_id, a)] = h
    return lookup


def _insert_interception_actions(df):
    interceptions = df.loc[df['type'].eq('Pass') & df['pass_type'].eq('Interception')].copy()
    if interceptions.empty:
        return df.assign(_synthetic_order=0.0, synthetic_action=False)
    base = df.assign(_synthetic_order=0.0, synthetic_action=False)
    interceptions['type'] = 'Interception'
    interceptions['_synthetic_order'] = -0.1
    interceptions['synthetic_action'] = True
    return pd.concat([base, interceptions], ignore_index=True, sort=False)


def convert_statsbomb_flat_to_spadl(ev, mt):
    match_meta = add_match_team_ids(mt, ev)
    opponent_lookup = build_opponent_lookup(match_meta)

    candidates = ev.loc[
        ev['team_id'].notna() & ev['type'].isin(SPADL_ACTION_RAW_TYPES | {'Pass'})
    ].copy()
    candidates = _insert_interception_actions(candidates)
    candidates['team_id'] = candidates['team_id'].astype(int)
    candidates['player_id'] = pd.to_numeric(candidates['player_id'], errors='coerce').astype('Int64')

    a = candidates.copy()
    a['type_name'] = 'non_action'
    a['result_name'] = 'success'
    a['bodypart_name'] = 'none'

    pass_m = a['type'].eq('Pass')
    a.loc[pass_m, 'type_name'] = 'pass'
    a.loc[pass_m & _truthy_series(a['pass_cross']).fillna(False), 'type_name'] = 'cross'
    a.loc[pass_m & a['pass_type'].eq('Throw-in'), 'type_name'] = 'throw_in'
    a.loc[pass_m & a['pass_type'].eq('Goal Kick'), 'type_name'] = 'goalkick'
    a.loc[pass_m & a['pass_type'].eq('Free Kick'), 'type_name'] = 'freekick_short'
    a.loc[pass_m & a['pass_type'].eq('Free Kick') & _truthy_series(a['pass_cross']).fillna(False), 'type_name'] = 'freekick_cross'
    a.loc[pass_m & a['pass_type'].eq('Corner'), 'type_name'] = 'corner_short'
    a.loc[pass_m & a['pass_type'].eq('Corner') & _truthy_series(a['pass_cross']).fillna(False), 'type_name'] = 'corner_cross'
    a.loc[pass_m & a['pass_outcome'].notna(), 'result_name'] = 'fail'
    a.loc[pass_m & a['pass_outcome'].eq('Pass Offside'), 'result_name'] = 'offside'
    a.loc[pass_m, 'bodypart_name'] = _normalize_bodypart(a.loc[pass_m, 'pass_body_part'])

    carry_m = a['type'].eq('Carry')
    a.loc[carry_m, 'type_name'] = 'dribble'
    a.loc[carry_m, 'result_name'] = 'success'
    a.loc[carry_m, 'bodypart_name'] = 'foot'

    take_on_m = a['type'].eq('Dribble')
    a.loc[take_on_m, 'type_name'] = 'take_on'
    a.loc[take_on_m, 'result_name'] = np.where(a.loc[take_on_m, 'dribble_outcome'].eq('Complete'), 'success', 'fail')
    a.loc[take_on_m, 'bodypart_name'] = 'foot'

    duel_m = a['type'].eq('Duel') & a['duel_type'].eq('Tackle')
    a.loc[duel_m, 'type_name'] = 'tackle'
    a.loc[duel_m, 'result_name'] = np.where(a.loc[duel_m, 'duel_outcome'].isin(SUCCESS_RESULTS), 'success', 'fail')

    int_m = a['type'].eq('Interception')
    a.loc[int_m, 'type_name'] = 'interception'
    a.loc[int_m, 'result_name'] = np.where(
        a.loc[int_m, 'interception_outcome'].isna() | a.loc[int_m, 'interception_outcome'].isin(SUCCESS_RESULTS),
        'success',
        'fail',
    )

    shot_m = a['type'].eq('Shot')
    a.loc[shot_m, 'type_name'] = 'shot'
    a.loc[shot_m & a['shot_type'].eq('Free Kick'), 'type_name'] = 'shot_freekick'
    a.loc[shot_m & a['shot_type'].eq('Penalty'), 'type_name'] = 'shot_penalty'
    a.loc[shot_m, 'result_name'] = np.where(a.loc[shot_m, 'shot_outcome'].eq('Goal'), 'success', 'fail')
    a.loc[shot_m, 'bodypart_name'] = _normalize_bodypart(a.loc[shot_m, 'shot_body_part'])

    gk_m = a['type'].eq('Goal Keeper')
    a.loc[gk_m & a['goalkeeper_type'].isin(SHOT_GOALKEEPER_TYPES), 'type_name'] = 'keeper_save'
    a.loc[gk_m & a['goalkeeper_type'].isin(CLAIM_GOALKEEPER_TYPES), 'type_name'] = 'keeper_claim'
    a.loc[gk_m & a['goalkeeper_type'].isin(PUNCH_GOALKEEPER_TYPES), 'type_name'] = 'keeper_punch'
    keeper_action_m = gk_m & a['type_name'].ne('non_action')
    a.loc[keeper_action_m, 'result_name'] = np.where(a.loc[keeper_action_m, 'goalkeeper_outcome'].isin(FAIL_GK), 'fail', 'success')
    a.loc[keeper_action_m, 'bodypart_name'] = _normalize_bodypart(a.loc[keeper_action_m, 'goalkeeper_body_part'])

    clearance_m = a['type'].eq('Clearance')
    a.loc[clearance_m, 'type_name'] = 'clearance'
    a.loc[clearance_m, 'bodypart_name'] = _normalize_bodypart(a.loc[clearance_m, 'clearance_body_part'])

    foul_m = a['type'].eq('Foul Committed')
    a.loc[foul_m, 'type_name'] = 'foul'
    a.loc[foul_m, 'result_name'] = 'fail'

    bad_touch_m = a['type'].eq('Miscontrol')
    a.loc[bad_touch_m, 'type_name'] = 'bad_touch'
    a.loc[bad_touch_m, 'result_name'] = 'fail'

    own_goal_m = a['type'].eq('Own Goal Against')
    a.loc[own_goal_m, 'type_name'] = 'bad_touch'
    a.loc[own_goal_m, 'result_name'] = 'owngoal'

    a = a.loc[a['type_name'].ne('non_action')].copy()

    a['start_x'] = _scale_x(a['location_x']).clip(0, SPADL_LENGTH)
    a['start_y'] = _scale_y(a['location_y']).clip(0, SPADL_WIDTH)
    a['end_x'] = a['start_x']
    a['end_y'] = a['start_y']

    pass_end_m = pass_m.loc[a.index] & a['pass_end_location_x'].notna() & a['pass_end_location_y'].notna()
    a.loc[pass_end_m, 'end_x'] = _scale_x(a.loc[pass_end_m, 'pass_end_location_x']).clip(0, SPADL_LENGTH)
    a.loc[pass_end_m, 'end_y'] = _scale_y(a.loc[pass_end_m, 'pass_end_location_y']).clip(0, SPADL_WIDTH)

    carry_end_m = a['type'].eq('Carry') & a['carry_end_location_x'].notna() & a['carry_end_location_y'].notna()
    a.loc[carry_end_m, 'end_x'] = _scale_x(a.loc[carry_end_m, 'carry_end_location_x']).clip(0, SPADL_LENGTH)
    a.loc[carry_end_m, 'end_y'] = _scale_y(a.loc[carry_end_m, 'carry_end_location_y']).clip(0, SPADL_WIDTH)

    shot_idx = a.index[a['type'].eq('Shot')]
    if len(shot_idx):
        sx = a.loc[shot_idx, 'shot_end_location'].map(lambda v: _list_item(v, 0))
        sy = a.loc[shot_idx, 'shot_end_location'].map(lambda v: _list_item(v, 1))
        a.loc[shot_idx, 'end_x'] = _scale_x(sx).fillna(SPADL_LENGTH).clip(0, SPADL_LENGTH)
        a.loc[shot_idx, 'end_y'] = _scale_y(sy).fillna(SPADL_WIDTH / 2).clip(0, SPADL_WIDTH)

    gk_idx = a.index[a['type_name'].isin(['keeper_save', 'keeper_claim', 'keeper_punch'])]
    if len(gk_idx):
        gx = a.loc[gk_idx, 'goalkeeper_end_location'].map(lambda v: _list_item(v, 0))
        gy = a.loc[gk_idx, 'goalkeeper_end_location'].map(lambda v: _list_item(v, 1))
        a.loc[gk_idx, 'end_x'] = _scale_x(gx).fillna(a.loc[gk_idx, 'start_x']).clip(0, SPADL_LENGTH)
        a.loc[gk_idx, 'end_y'] = _scale_y(gy).fillna(a.loc[gk_idx, 'start_y']).clip(0, SPADL_WIDTH)

    a['time_seconds'] = pd.to_numeric(a['minute'], errors='coerce').fillna(0) * 60 + pd.to_numeric(a['second'], errors='coerce').fillna(0)
    a['period_id'] = pd.to_numeric(a['period'], errors='coerce').fillna(0).astype(int)
    a['event_minute'] = a['time_seconds'] / 60.0
    a['original_event_id'] = a['id']
    a['original_index'] = pd.to_numeric(a['index'], errors='coerce')

    a['goal_team_id'] = pd.NA
    a.loc[a['type'].eq('Shot') & a['shot_outcome'].eq('Goal'), 'goal_team_id'] = a.loc[a['type'].eq('Shot') & a['shot_outcome'].eq('Goal'), 'team_id']
    own_goal_rows = a.index[a['result_name'].eq('owngoal')]
    if len(own_goal_rows):
        a.loc[own_goal_rows, 'goal_team_id'] = [opponent_lookup.get((r.match_id, int(r.team_id))) for r in a.loc[own_goal_rows].itertuples(index=False)]
    a['goal_team_id'] = pd.to_numeric(a['goal_team_id'], errors='coerce').astype('Int64')
    a['is_goal'] = a['goal_team_id'].notna()

    a = a.merge(match_meta[['match_id', 'home_team_id', 'away_team_id']], on='match_id', how='left')
    a = a.sort_values(['match_id', 'period_id', 'time_seconds', 'original_index', '_synthetic_order']).reset_index(drop=True)
    a['action_id'] = np.arange(len(a), dtype=np.int64)

    keep_cols = [
        'action_id', 'original_event_id', 'synthetic_action', 'match_id', 'period_id', 'minute', 'second',
        'timestamp', 'time_seconds', 'event_minute', 'original_index', 'index', 'type', 'type_name',
        'result_name', 'bodypart_name', 'team', 'team_id', 'player', 'player_id', 'position', 'play_pattern',
        'home_team_id', 'away_team_id', 'start_x', 'start_y', 'end_x', 'end_y', 'shot_statsbomb_xg',
        'shot_outcome', 'goalkeeper_type', 'goalkeeper_outcome', 'goal_team_id', 'is_goal',
    ]
    keep_cols = [c for c in keep_cols if c in a.columns]
    return a[keep_cols]


def add_score_context(actions_df):
    a = actions_df.copy()
    is_home_goal = a['goal_team_id'].notna() & a['goal_team_id'].eq(a['home_team_id'])
    is_away_goal = a['goal_team_id'].notna() & a['goal_team_id'].eq(a['away_team_id'])
    a['_home_goal'] = is_home_goal.astype(int)
    a['_away_goal'] = is_away_goal.astype(int)
    grp = a.groupby('match_id', sort=False)
    a['home_score_before'] = grp['_home_goal'].cumsum() - a['_home_goal']
    a['away_score_before'] = grp['_away_goal'].cumsum() - a['_away_goal']
    a['home_score_after'] = a['home_score_before'] + a['_home_goal']
    a['away_score_after'] = a['away_score_before'] + a['_away_goal']
    is_home_action = a['team_id'].eq(a['home_team_id'])
    a['score_diff_before'] = np.where(
        is_home_action,
        a['home_score_before'] - a['away_score_before'],
        a['away_score_before'] - a['home_score_before'],
    ).astype('int16')
    a['score_diff_after'] = np.where(
        is_home_action,
        a['home_score_after'] - a['away_score_after'],
        a['away_score_after'] - a['home_score_after'],
    ).astype('int16')
    return a.drop(columns=['_home_goal', '_away_goal'])


actions = add_score_context(convert_statsbomb_flat_to_spadl(events, matches))
excluded_counts = events['type'].value_counts().reindex(sorted(EXCLUDED_DEFENSIVE_CONTEXT_TYPES)).fillna(0).astype(int)
print(f'Ações SPADL-like: {len(actions):,}  |  Tipos VAEP: {actions["type_name"].nunique()}')
print('Eventos StatsBomb contextuais excluídos do VAEP base:')
print(excluded_counts.to_string())
actions[['type', 'type_name', 'result_name', 'bodypart_name']].value_counts().head(20)


## **5. Labels de Gol Futuro**

Para cada ação $a_i$, `label_score = 1` se o time da ação marcar nas próximas `HORIZON` ações; `label_concede = 1` se o adversário marcar. Gols contra usam `goal_team_id` do adversário para manter a perspectiva correta.


In [ ]:
def add_future_goal_labels(df, horizon=10):
    df = df.copy()
    scores   = np.zeros(len(df), dtype=bool)
    concedes = np.zeros(len(df), dtype=bool)
    teams    = df['team_id'].astype(int).to_numpy()
    goal_team = df['goal_team_id'].fillna(-1).astype(int).to_numpy()

    for indices in df.groupby('match_id', sort=False).indices.values():
        idx = np.asarray(indices)
        g_teams = teams[idx]
        g_goal_team = goal_team[idx]
        g_scores = np.zeros(len(idx), dtype=bool)
        g_concedes = np.zeros(len(idx), dtype=bool)
        for step in range(horizon):
            if step >= len(idx):
                break
            future_goal_team = np.full(len(idx), -1, dtype=int)
            future_goal_team[:len(idx)-step] = g_goal_team[step:]
            has_goal = future_goal_team != -1
            g_scores |= has_goal & (future_goal_team == g_teams)
            g_concedes |= has_goal & (future_goal_team != g_teams)
        scores[idx] = g_scores
        concedes[idx] = g_concedes

    df['label_score'] = scores.astype(int)
    df['label_concede'] = concedes.astype(int)
    return df

actions = add_future_goal_labels(actions, HORIZON)
print(f'Taxa score:   {actions["label_score"].mean():.4f}')
print(f'Taxa concede: {actions["label_concede"].mean():.4f}')


## **6. Features de Estado**

O estado $s_i$ usa as últimas 3 ações em orientação esquerda→direita para o time da ação. As features seguem o espírito do `socceraction`: tipo de ação, resultado, parte do corpo, localizações inicial/final, deslocamento, deltas temporais, distância/ângulo ao gol e placar. `team_id` não entra como feature para reduzir viés de força do time.


In [ ]:
CAT_FEATURES = ['type_name', 'result_name', 'bodypart_name', 'play_pattern', 'period_id']
NUM_FEATURES = [
    'start_x_lr', 'start_y_lr', 'end_x_lr', 'end_y_lr',
    'dx_lr', 'dy_lr', 'movement', 'time_seconds', 'time_delta',
    'distance_start_goal', 'angle_start_goal', 'distance_end_goal', 'angle_end_goal',
    'score_diff_after',
]
BASE_FEATURES = CAT_FEATURES + NUM_FEATURES


def _goal_distance(x, y):
    return np.sqrt((SPADL_LENGTH - x) ** 2 + (SPADL_WIDTH / 2 - y) ** 2)


def _goal_angle(x, y):
    goal_y1 = SPADL_WIDTH / 2 - GOAL_WIDTH / 2
    goal_y2 = SPADL_WIDTH / 2 + GOAL_WIDTH / 2
    dx = np.maximum(SPADL_LENGTH - x, 1e-6)
    a1 = np.arctan2(goal_y1 - y, dx)
    a2 = np.arctan2(goal_y2 - y, dx)
    return np.abs(a2 - a1)


def orient_actions_left_to_right(df):
    o = df.copy()
    is_home = o['team_id'].eq(o['home_team_id'])
    for base, lr in [('start_x', 'start_x_lr'), ('end_x', 'end_x_lr')]:
        o[lr] = np.where(is_home, o[base], SPADL_LENGTH - o[base])
    for base, lr in [('start_y', 'start_y_lr'), ('end_y', 'end_y_lr')]:
        o[lr] = np.where(is_home, o[base], SPADL_WIDTH - o[base])
    o['dx_lr'] = o['end_x_lr'] - o['start_x_lr']
    o['dy_lr'] = o['end_y_lr'] - o['start_y_lr']
    o['movement'] = np.sqrt(o['dx_lr'] ** 2 + o['dy_lr'] ** 2)
    grp = o.groupby('match_id', sort=False)
    o['time_delta'] = grp['time_seconds'].diff().clip(lower=0, upper=60).fillna(0)
    o['distance_start_goal'] = _goal_distance(o['start_x_lr'], o['start_y_lr'])
    o['angle_start_goal'] = _goal_angle(o['start_x_lr'], o['start_y_lr'])
    o['distance_end_goal'] = _goal_distance(o['end_x_lr'], o['end_y_lr'])
    o['angle_end_goal'] = _goal_angle(o['end_x_lr'], o['end_y_lr'])
    return o


def build_state_features(df):
    feat_df = orient_actions_left_to_right(df)
    grouped = feat_df.groupby('match_id', sort=False)
    x = pd.DataFrame(index=feat_df.index)
    for lag in range(3):
        shifted = feat_df[BASE_FEATURES] if lag == 0 else grouped[BASE_FEATURES].shift(lag)
        for col in CAT_FEATURES:
            x[f'{col}_lag{lag}'] = shifted[col].astype('string').fillna('missing').astype('category')
        for col in NUM_FEATURES:
            x[f'{col}_lag{lag}'] = pd.to_numeric(shifted[col], errors='coerce').fillna(-1).astype('float32')
    x['same_team_lag1'] = feat_df['team_id'].eq(grouped['team_id'].shift(1)).fillna(False).astype('int8')
    x['same_team_lag2'] = feat_df['team_id'].eq(grouped['team_id'].shift(2)).fillna(False).astype('int8')
    return x

X = build_state_features(actions)
print(f'Features: {X.shape[1]}  |  Amostras: {X.shape[0]:,}')


## **7. Modelos VAEP**

Dois classificadores XGBoost: $P_{score}$ e $P_{concede}$. Divisão treino/validação por partida (sem vazamento). Modelo final treinado em todos os dados.

$$\text{VAEP}(a_i) = \bigl[P_{score}(s_{i+1}) - P_{score}(s_i)\bigr] - \bigl[P_{concede}(s_{i+1}) - P_{concede}(s_i)\bigr]$$

In [ ]:
def _safe_binary_metrics(y_true, p):
    metrics = {
        'log_loss': log_loss(y_true, p, labels=[0, 1]),
        'brier': brier_score_loss(y_true, p),
    }
    metrics['roc_auc'] = roc_auc_score(y_true, p) if y_true.nunique() == 2 else np.nan
    return metrics


def fit_vaep(actions_df, X_features):
    y_score   = actions_df['label_score'].astype(int)
    y_concede = actions_df['label_concede'].astype(int)
    groups    = actions_df['match_id']

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    tr, va   = next(splitter.split(X_features, y_score, groups=groups))

    score_val   = XGBClassifier(**XGB_PARAMS).fit(X_features.iloc[tr], y_score.iloc[tr])
    concede_val = XGBClassifier(**XGB_PARAMS).fit(X_features.iloc[tr], y_concede.iloc[tr])
    p_score_val = score_val.predict_proba(X_features.iloc[va])[:, 1]
    p_concede_val = concede_val.predict_proba(X_features.iloc[va])[:, 1]
    score_metrics = _safe_binary_metrics(y_score.iloc[va], p_score_val)
    concede_metrics = _safe_binary_metrics(y_concede.iloc[va], p_concede_val)

    val_metrics = pd.DataFrame([
        {'model': 'P_score', 'positive_rate': y_score.iloc[va].mean(), **score_metrics},
        {'model': 'P_concede', 'positive_rate': y_concede.iloc[va].mean(), **concede_metrics},
    ])

    score_full   = XGBClassifier(**XGB_PARAMS).fit(X_features, y_score)
    concede_full = XGBClassifier(**XGB_PARAMS).fit(X_features, y_concede)

    a = actions_df.copy()
    a['p_score_after']   = score_full.predict_proba(X_features)[:, 1]
    a['p_concede_after'] = concede_full.predict_proba(X_features)[:, 1]

    grp = a.groupby('match_id', sort=False)
    a['prev_team_id'] = grp['team_id'].shift(1)
    a['prev_period_id'] = grp['period_id'].shift(1)
    a['same_team_as_previous'] = a['team_id'].eq(a['prev_team_id'])
    a['new_restart_state'] = a['prev_team_id'].isna() | a['period_id'].ne(a['prev_period_id']) | grp['is_goal'].shift(1).fillna(False)
    a['prev_p_score'] = grp['p_score_after'].shift(1).fillna(0)
    a['prev_p_concede'] = grp['p_concede_after'].shift(1).fillna(0)
    a['p_score_before'] = np.where(a['same_team_as_previous'], a['prev_p_score'], a['prev_p_concede'])
    a['p_concede_before'] = np.where(a['same_team_as_previous'], a['prev_p_concede'], a['prev_p_score'])
    a.loc[a['new_restart_state'], ['p_score_before', 'p_concede_before']] = 0.0
    a['vaep'] = (a['p_score_after'] - a['p_score_before']) - (a['p_concede_after'] - a['p_concede_before'])

    cols = [
        'action_id', 'original_event_id', 'synthetic_action', 'match_id', 'period_id', 'minute', 'second',
        'event_minute', 'time_seconds', 'index', 'original_index', 'type', 'type_name', 'result_name',
        'bodypart_name', 'team', 'team_id', 'player', 'player_id', 'position', 'play_pattern',
        'home_team_id', 'away_team_id', 'start_x', 'start_y', 'end_x', 'end_y', 'score_diff_before',
        'score_diff_after', 'goal_team_id', 'is_goal', 'p_score_before', 'p_score_after',
        'p_concede_before', 'p_concede_after', 'vaep', 'label_score', 'label_concede',
    ]
    cols = [c for c in cols if c in a.columns]
    vaep_df = a[cols].copy()
    vaep_df.to_parquet(PROCESSED_PATH / 'vaep_actions.parquet', index=False)
    val_metrics.to_parquet(PROCESSED_PATH / 'vaep_validation.parquet', index=False)
    with open(PROCESSED_PATH / 'xgb_score_model.pkl', 'wb') as f:
        pickle.dump(score_full, f)
    with open(PROCESSED_PATH / 'xgb_concede_model.pkl', 'wb') as f:
        pickle.dump(concede_full, f)
    with open(PROCESSED_PATH / 'vaep_cache_metadata.json', 'w') as f:
        json.dump({
            'version': VAEP_CACHE_VERSION,
            'horizon': HORIZON,
            'n_actions': int(len(vaep_df)),
            'feature_count': int(X_features.shape[1]),
            'excluded_context_events': sorted(EXCLUDED_DEFENSIVE_CONTEXT_TYPES),
            'uses_team_id_feature': False,
        }, f, indent=2)

    return vaep_df, val_metrics


In [ ]:
vaep_path = PROCESSED_PATH / 'vaep_actions.parquet'
val_path = PROCESSED_PATH / 'vaep_validation.parquet'
meta_path = PROCESSED_PATH / 'vaep_cache_metadata.json'

cache_current = False
if vaep_path.exists() and val_path.exists() and meta_path.exists():
    with open(meta_path) as f:
        cache_meta = json.load(f)
    cache_current = cache_meta.get('version') == VAEP_CACHE_VERSION and cache_meta.get('horizon') == HORIZON

if cache_current and not RECOMPUTE_VAEP:
    vaep_actions = pd.read_parquet(vaep_path)
    val_metrics  = pd.read_parquet(val_path)
else:
    vaep_actions, val_metrics = fit_vaep(actions, X)

print(f'Ações com VAEP: {len(vaep_actions):,}')
print(f'Cache VAEP atual: {cache_current and not RECOMPUTE_VAEP}')
val_metrics


In [ ]:
def parse_tactics(value):
    if isinstance(value, dict): return value
    if value is None or (isinstance(value, float) and np.isnan(value)): return {}
    if isinstance(value, str) and value:
        try: return ast.literal_eval(value)
        except (ValueError, SyntaxError): return {}
    return {}

def lineup_from_tactics(value):
    tactics = parse_tactics(value)
    players, names = set(), {}
    for item in tactics.get('lineup', []):
        player = item.get('player', {})
        pid = player.get('id')
        if pid is not None:
            players.add(int(pid))
            names[int(pid)] = player.get('name')
    return players, names

def _starting_slots(value):
    tactics = parse_tactics(value)
    slots = {}
    for slot_id, item in enumerate(tactics.get('lineup', [])):
        player   = item.get('player', {})
        position = item.get('position', {})
        pid = player.get('id')
        if pid is None: continue
        pid = int(pid)
        slots[int(slot_id)] = {
            'starting_player_id': pid,
            'starting_position': position.get('name'),
            'current_player_id': pid,
            'players': {pid},
        }
    return slots

def _slot_players(slots):
    return tuple((int(s), int(d['current_player_id'])) for s, d in sorted(slots.items()) if d.get('current_player_id') is not None)

def _find_slot(slots, pid):
    for s, d in slots.items():
        if d.get('current_player_id') == int(pid): return int(s)
    return None

def _extra_slot(slots, pid=None):
    s = max(slots.keys(), default=-1) + 1
    slots[s] = {'starting_player_id': None, 'current_player_id': int(pid) if pid else None, 'players': {int(pid)} if pid else set()}
    return s

def _append_seg(rows, match_id, season, match_date, team_id, team, t0, t1, slots):
    sp = _slot_players(slots)
    active = frozenset(p for _, p in sp)
    if not active or t1 <= t0: return
    rows.append({'match_id':match_id,'season':season,'match_date':match_date,
                 'team_id':team_id,'team':team,'start_minute':t0,'end_minute':t1,
                 'duration':t1-t0,'active':active,'slot_players':sp})

def build_presence_segments(ev, mt):
    segs = []
    match_end  = ev.groupby('match_id')['event_minute'].max().to_dict()
    match_meta = mt[['match_id','season','match_date']].drop_duplicates('match_id').set_index('match_id')
    state_evs  = {'Substitution','Player Off','Player On'}

    for (match_id, team_id), grp in ev.dropna(subset=['team_id']).groupby(['match_id','team_id'], sort=False):
        team_id = int(team_id)
        start_rows = grp.loc[grp['type'].eq('Starting XI')]
        if start_rows.empty: continue
        slots = _starting_slots(start_rows.iloc[0]['tactics'])
        if not slots: continue
        p2s = {int(d['current_player_id']): int(s) for s, d in slots.items() if d.get('current_player_id') is not None}
        tmp_off = {}
        last   = 0.0
        end    = float(match_end.get(match_id, grp['event_minute'].max()))
        team   = grp['team'].dropna().iloc[0] if grp['team'].notna().any() else None
        season = match_meta.loc[match_id,'season'] if match_id in match_meta.index else None
        mdate  = pd.to_datetime(match_meta.loc[match_id,'match_date']) if match_id in match_meta.index else pd.NaT
        changes = grp.loc[grp['type'].isin(state_evs)].sort_values(['event_minute','index'])
        match_segs = []

        for row in changes.itertuples(index=False):
            cur = float(row.event_minute)
            _append_seg(match_segs, match_id, season, mdate, team_id, team, last, cur, slots)
            if row.type == 'Substitution':
                off_id = int(row.player_id) if pd.notna(row.player_id) else None
                on_id  = int(row.substitution_replacement_id) if pd.notna(row.substitution_replacement_id) else None
                sid = (_find_slot(slots, off_id) or tmp_off.get(off_id) or p2s.get(off_id)) if off_id else None
                if sid is None:
                    sid = _extra_slot(slots, on_id)
                    if on_id: p2s[on_id] = int(sid)
                else:
                    tmp_off.pop(off_id, None)
                    slots[sid]['current_player_id'] = on_id
                    if on_id: slots[sid]['players'].add(on_id); p2s[on_id] = int(sid)
            elif row.type == 'Player Off' and pd.notna(row.player_id):
                pid = int(row.player_id)
                sid = _find_slot(slots, pid)
                if sid is not None: slots[sid]['current_player_id'] = None; tmp_off[pid] = int(sid)
            elif row.type == 'Player On' and pd.notna(row.player_id):
                pid = int(row.player_id)
                sid = tmp_off.pop(pid, None) or p2s.get(pid)
                if sid is None: sid = _extra_slot(slots, pid)
                else: slots[sid]['current_player_id'] = pid; slots[sid]['players'].add(pid); p2s[pid] = int(sid)
            last = max(last, cur)

        _append_seg(match_segs, match_id, season, mdate, team_id, team, last, end, slots)

        starter_slots = tuple(
            (int(s), int(d['starting_player_id']), d.get('starting_position'))
            for s, d in sorted(slots.items()) if d.get('starting_player_id') is not None
        )
        match_player_slots = tuple(
            (int(pid), int(s)) for s, d in sorted(slots.items()) for pid in sorted(d['players'])
        )
        match_participants = frozenset(pid for pid, _ in match_player_slots)
        for seg in match_segs:
            seg['starter_slots']      = starter_slots
            seg['match_player_slots'] = match_player_slots
            seg['match_participants'] = match_participants
            segs.append(seg)

    return pd.DataFrame(segs)

def player_minutes_per_match(segs):
    rows = []
    for seg in segs.itertuples(index=False):
        for pid in seg.active:
            rows.append({'match_id':seg.match_id,'season':seg.season,'match_date':seg.match_date,
                         'team_id':int(seg.team_id),'team':seg.team,'player_id':int(pid),'minutes':seg.duration})
    df = pd.DataFrame(rows)
    return df.groupby(['match_id','season','match_date','team_id','team','player_id'], as_index=False)['minutes'].sum()

def build_lineup_slot_segments(segs):
    rows = []
    for seg in segs.itertuples(index=False):
        by_slot    = {int(s): int(p) for s,p,_ in seg.starter_slots}
        by_slot_pos= {int(s): pos for s,_,pos in seg.starter_slots}
        for sid, pid in seg.slot_players:
            rows.append({'match_id':seg.match_id,'season':seg.season,'match_date':seg.match_date,
                         'team_id':int(seg.team_id),'team':seg.team,'slot_id':int(sid),
                         'slot_starter_id':by_slot.get(int(sid)),'slot_starter_position':by_slot_pos.get(int(sid)),
                         'player_id':int(pid),'start_minute':float(seg.start_minute),
                         'end_minute':float(seg.end_minute),'duration':float(seg.duration)})
    return pd.DataFrame(rows)

In [ ]:
segments = build_presence_segments(events, matches)
lineup_slot_segs = build_lineup_slot_segments(segments)
lineup_slot_segs.to_parquet(PROCESSED_PATH / 'lineup_slot_segments.parquet', index=False)
player_min = player_minutes_per_match(segments)
print(f'Segmentos: {len(segments):,}  |  Registros de minutos: {len(player_min):,}')

## **8. VAEP por Jogador**

In [ ]:
def extract_goalkeeper_ids(ev, vaep):
    gk = set()
    for row in ev[ev['type'] == 'Starting XI'].itertuples(index=False):
        tactics = parse_tactics(row.tactics)
        for item in tactics.get('lineup', []):
            if item.get('position', {}).get('name') == 'Goalkeeper':
                pid = item.get('player', {}).get('id')
                if pid: gk.add(int(pid))
    gk |= set(vaep.loc[vaep['type'].eq('Goal Keeper'), 'player_id'].dropna().astype(int).unique())
    return gk

def build_player_lookup(ev):
    rows = []
    direct = ev.loc[ev['player_id'].notna(), ['team_id','team','player_id','player']].dropna(subset=['team_id','player_id'])
    rows.append(direct)
    subs = ev.loc[ev['substitution_replacement_id'].notna(), ['team_id','team','substitution_replacement_id','substitution_replacement']].dropna(subset=['team_id'])
    subs = subs.rename(columns={'substitution_replacement_id':'player_id','substitution_replacement':'player'})
    rows.append(subs)
    lineup_rows = []
    for row in ev.loc[ev['type'].eq('Starting XI'), ['team_id','team','tactics']].itertuples(index=False):
        players, names = lineup_from_tactics(row.tactics)
        for pid in players:
            lineup_rows.append({'team_id':row.team_id,'team':row.team,'player_id':pid,'player':names.get(pid)})
    rows.append(pd.DataFrame(lineup_rows))
    lk = pd.concat(rows, ignore_index=True).dropna(subset=['team_id','player_id','player'])
    lk['team_id']   = lk['team_id'].astype(int)
    lk['player_id'] = lk['player_id'].astype(int)
    return lk.drop_duplicates(['team_id','player_id'], keep='last')

goalkeeper_ids = extract_goalkeeper_ids(events, vaep_actions)
player_lookup  = build_player_lookup(events)
print(f'Goleiros: {len(goalkeeper_ids)}  |  Jogadores no lookup: {len(player_lookup):,}')

In [ ]:
vaep_by_pm = (
    vaep_actions.loc[vaep_actions['player_id'].notna()]
    .assign(team_id=lambda d: d['team_id'].astype(int), player_id=lambda d: d['player_id'].astype(int))
    .groupby(['match_id','team_id','player_id'], as_index=False)
    .agg(vaep_sum=('vaep','sum'), n_actions=('vaep','count'))
)

player_match = player_min.merge(vaep_by_pm, on=['match_id','team_id','player_id'], how='left')
player_match['vaep_sum']  = player_match['vaep_sum'].fillna(0.0)
player_match['n_actions'] = player_match['n_actions'].fillna(0).astype(int)

player_quality = (
    player_match[~player_match['player_id'].isin(goalkeeper_ids)]
    .groupby('player_id')
    .agg(vaep_total=('vaep_sum','sum'), min_total=('minutes','sum'), actions=('n_actions','sum'))
    .merge(player_lookup[['player_id','player','team','team_id']].drop_duplicates('player_id'), on='player_id', how='left')
    .merge(matches[['match_id','competition']].drop_duplicates().merge(
        player_min[['match_id','player_id']], on='match_id').groupby('player_id')['competition'].agg(lambda x: x.mode()[0]).reset_index(),
        on='player_id', how='left')
    .reset_index()
)

player_quality.head(5)


In [ ]:
total_vaep = player_quality.sort_values('vaep_total', ascending=False)

print(f'Total de jogadores (excl. goleiros): {len(total_vaep):,}')
total_vaep[['player','team','competition','min_total','vaep_total','actions']].head(20).round(3)

In [ ]:
top10_total = total_vaep.head(10).copy()
top10_total['label'] = top10_total['player'].str.split().str[-1] + ' (' + top10_total['team'].fillna('') + ')'

bot10_total = total_vaep.tail(10).copy()
bot10_total['label'] = bot10_total['player'].str.split().str[-1] + ' (' + bot10_total['team'].fillna('') + ')'

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top10_total['label'][::-1], top10_total['vaep_total'][::-1], color='#1b9e77', edgecolor='black', linewidth=0.6)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('VAEP Total')
axes[0].set_title('Top-10 Jogadores — VAEP Total')

axes[1].barh(bot10_total['label'][::-1], bot10_total['vaep_total'][::-1], color='#d95f02', edgecolor='black', linewidth=0.6)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('VAEP Total')
axes[1].set_title('Bottom-10 Jogadores — VAEP Total')

plt.tight_layout()
fig.savefig(VIZ_PATH / 'vaep_total_players.png', dpi=150, bbox_inches='tight')
plt.show()

## **9. VAEP por Time**

In [ ]:
team_vaep = (
    player_match[~player_match['player_id'].isin(goalkeeper_ids)]
    .groupby(['team_id','team'], as_index=False)
    .agg(vaep_total=('vaep_sum','sum'), min_total=('minutes','sum'), actions=('n_actions','sum'))
    .sort_values('vaep_total', ascending=False)
    .reset_index(drop=True)
)

team_vaep.head(20).round(3)


In [ ]:
top10_teams = team_vaep.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10_teams['team'][::-1], top10_teams['vaep_total'][::-1], color='#2166ac', edgecolor='black', linewidth=0.6)
ax.set_xlabel('VAEP Total')
ax.set_title('Top-10 Times — VAEP Total')
plt.tight_layout()
fig.savefig(VIZ_PATH / 'vaep_teams.png', dpi=150, bbox_inches='tight')
plt.show()

## **10. VAEP por Liga**

In [ ]:
match_competition = matches[['match_id','competition']].drop_duplicates()

league_vaep = (
    player_match[~player_match['player_id'].isin(goalkeeper_ids)]
    .merge(match_competition, on='match_id', how='left')
    .groupby('competition', as_index=False)
    .agg(vaep_total=('vaep_sum','sum'), min_total=('minutes','sum'), actions=('n_actions','sum'), n_players=('player_id','nunique'))
    .sort_values('vaep_total', ascending=False)
    .reset_index(drop=True)
)

league_vaep.round(3)


In [ ]:
top10_leagues = league_vaep.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10_leagues['competition'][::-1], top10_leagues['vaep_total'][::-1], color='#7b2d8b', edgecolor='black', linewidth=0.6)
ax.set_xlabel('VAEP Total')
ax.set_title('VAEP Total por Liga')
plt.tight_layout()
fig.savefig(VIZ_PATH / 'vaep_leagues.png', dpi=150, bbox_inches='tight')
plt.show()

---
## **Parte 2 — Synergy VAEP**

Sinergia baseada em proxy real de ausência. Para cada par (A, B) do mesmo time:

- **Juntos**: minutos em que A e B estiveram em campo simultaneamente.
- **Proxy de B**: nos jogos em que B ficou fora, usa o ocupante real do mesmo slot de substituição ou o titular mais similar espacialmente.

**Score de sinergia:** `VAEP90(A+B juntos) − VAEP90(A+proxy_de_B)`

## **11. Minutagem por Jogo — Todos os Jogadores do Elenco**

Para cada time e cada partida disputada, registra os minutos de **todos** os jogadores que fizeram parte do elenco naquela temporada/competição.  
Jogadores que não entraram em campo recebem `minutes = 0`.

Colunas de output: `team_id`, `team`, `match_id`, `player_id`, `player`, `minutes`, `started`, `subbed_out`, `sub_replacement_id`, `came_as_sub`

In [ ]:
def build_team_rosters(ev):
    direct = ev.loc[
        ev['player_id'].notna() & ev['team_id'].notna(),
        ['team_id', 'team', 'player_id', 'player']
    ].dropna(subset=['player_id']).copy()
    direct['player_id'] = direct['player_id'].astype(int)
    direct['team_id']   = direct['team_id'].astype(int)

    subs = ev.loc[
        ev['substitution_replacement_id'].notna() & ev['team_id'].notna(),
        ['team_id', 'team', 'substitution_replacement_id', 'substitution_replacement']
    ].rename(columns={
        'substitution_replacement_id': 'player_id',
        'substitution_replacement':    'player'
    }).dropna(subset=['player_id']).copy()
    subs['player_id'] = subs['player_id'].astype(int)
    subs['team_id']   = subs['team_id'].astype(int)

    combined = pd.concat([direct, subs], ignore_index=True)
    return combined.drop_duplicates(['team_id', 'player_id']).reset_index(drop=True)


def build_full_player_match_minutes(ev, pm, rosters):
    subs = ev.loc[
        ev['type'].eq('Substitution') &
        ev['player_id'].notna() &
        ev['substitution_replacement_id'].notna(),
        ['match_id', 'team_id', 'player_id', 'substitution_replacement_id']
    ].copy()
    subs['player_id']                   = subs['player_id'].astype(int)
    subs['substitution_replacement_id'] = subs['substitution_replacement_id'].astype(int)
    subs['team_id']                     = subs['team_id'].astype(int)

    sub_out_map = {}
    sub_in_set  = set()
    for row in subs.itertuples(index=False):
        sub_out_map[(row.match_id, row.team_id, row.player_id)] = int(row.substitution_replacement_id)
        sub_in_set.add((row.match_id, row.team_id, int(row.substitution_replacement_id)))

    starting_xi = {}
    for row in ev.loc[ev['type'].eq('Starting XI'), ['match_id', 'team_id', 'tactics']].itertuples(index=False):
        tactics = parse_tactics(row.tactics)
        pids = {int(item['player']['id']) for item in tactics.get('lineup', [])
                if item.get('player', {}).get('id') is not None}
        starting_xi[(row.match_id, int(row.team_id))] = pids

    pm_idx = (
        pm.assign(team_id=lambda d: d['team_id'].astype(int), player_id=lambda d: d['player_id'].astype(int))
        .set_index(['match_id', 'team_id', 'player_id'])['minutes']
        .to_dict()
    )

    team_matches = (
        pm[['match_id', 'team_id', 'team']].drop_duplicates()
        .assign(team_id=lambda d: d['team_id'].astype(int))
    )

    roster_by_team = {
        int(tid): grp for tid, grp in rosters.groupby('team_id')
    }

    rows = []
    for rec in team_matches.itertuples(index=False):
        match_id, team_id, team = rec.match_id, int(rec.team_id), rec.team
        roster_grp = roster_by_team.get(team_id)
        if roster_grp is None:
            continue
        starters = starting_xi.get((match_id, team_id), set())

        for pr in roster_grp.itertuples(index=False):
            pid     = int(pr.player_id)
            minutes = float(pm_idx.get((match_id, team_id, pid), 0.0))
            started = pid in starters
            repl_id = sub_out_map.get((match_id, team_id, pid))
            rows.append({
                'match_id':          match_id,
                'team_id':           team_id,
                'team':              team,
                'player_id':         pid,
                'player':            pr.player,
                'minutes':           minutes,
                'started':           started,
                'subbed_out':        repl_id is not None,
                'sub_replacement_id': repl_id,
                'came_as_sub':       (match_id, team_id, pid) in sub_in_set,
            })

    return pd.DataFrame(rows)


team_rosters          = build_team_rosters(events)
full_player_match_min = build_full_player_match_minutes(events, player_min, team_rosters)

print(f'Registros totais (jogador × partida): {len(full_player_match_min):,}')
print(f'  Com 0 minutos (não entrou):  {full_player_match_min["minutes"].eq(0).sum():,}')
print(f'  Titulares:                   {full_player_match_min["started"].sum():,}')
print(f'  Substituídos para fora:      {full_player_match_min["subbed_out"].sum():,}')
print(f'  Entraram como substitutos:   {full_player_match_min["came_as_sub"].sum():,}')


In [ ]:
sample_team_id = int(full_player_match_min['team_id'].value_counts().idxmax())
sample_matches = (
    full_player_match_min[full_player_match_min['team_id'].eq(sample_team_id)]
    ['match_id'].unique()[:3]
)

(
    full_player_match_min[
        full_player_match_min['team_id'].eq(sample_team_id) &
        full_player_match_min['match_id'].isin(sample_matches)
    ]
    .sort_values(['match_id', 'minutes'], ascending=[True, False])
    [['team', 'match_id', 'player', 'minutes', 'started', 'subbed_out', 'came_as_sub']]
    .head(30)
)


## **12. Matriz de Ocorrências Espaciais (14 × 10)**

Para cada jogador com pelo menos 600 minutos totais, conta-se o número de eventos em um grid **14 × 10** (120 m × 80 m) e normaliza-se pelo total de eventos do jogador.  
O vetor resultante (**140 dimensões**) representa o perfil espacial de cada jogador.

PCA na formulação snapshot/SVD — mesma metodologia do TCC (eigen-heatmaps) — reduz para `k` componentes que explicam ≥ 90 % da variância, sem teto manual de componentes.  
Similaridade entre jogadores via **distância cosseno** no espaço reduzido.

In [ ]:
OMAT_X_BINS          = 14
OMAT_Y_BINS          = 10
OCC_VARIANCE_TARGET  = 0.90
MIN_EIGENMAP_MINUTES = 600.0

NON_SPATIAL_TYPES = {
    'Starting XI', 'Half Start', 'Half End', 'Tactical Shift',
    'Substitution', 'Player Off', 'Player On', 'Injury Stoppage',
    'Bad Behaviour', 'Referee Ball-Drop',
}

def build_occurrence_matrix(ev, player_qual, gk_ids, x_bins=14, y_bins=10):
    eligible_players = set(
        player_qual.loc[player_qual['min_total'].ge(MIN_EIGENMAP_MINUTES), 'player_id']
        .dropna()
        .astype(int)
    )

    spatial = ev.loc[
        ev['team_id'].notna()    & ev['player_id'].notna() &
        ev['location_x'].notna() & ev['location_y'].notna() &
        ~ev['type'].isin(NON_SPATIAL_TYPES),
        ['player_id', 'team_id', 'team', 'player', 'location_x', 'location_y']
    ].copy()
    spatial['player_id']  = spatial['player_id'].astype(int)
    spatial['team_id']    = spatial['team_id'].astype(int)
    spatial = spatial[~spatial['player_id'].isin(gk_ids)]
    spatial = spatial[spatial['player_id'].isin(eligible_players)].copy()
    spatial['location_x'] = pd.to_numeric(spatial['location_x'], errors='coerce')
    spatial['location_y'] = pd.to_numeric(spatial['location_y'], errors='coerce')
    spatial = spatial[
        spatial['location_x'].between(0, 120, inclusive='both') &
        spatial['location_y'].between(0, 80,  inclusive='both')
    ].copy()

    x_edges = np.linspace(0, 120, x_bins + 1)
    y_edges = np.linspace(0, 80,  y_bins + 1)
    spatial['x_bin'] = pd.cut(spatial['location_x'], bins=x_edges, labels=False,
                               include_lowest=True).astype(int)
    spatial['y_bin'] = pd.cut(spatial['location_y'], bins=y_edges, labels=False,
                               include_lowest=True).astype(int)
    spatial['cell']  = spatial['x_bin'] * y_bins + spatial['y_bin']

    player_info = (
        spatial.groupby('player_id', as_index=False)
        .agg(player=('player', 'first'), team=('team', 'first'),
             team_id=('team_id', 'first'), n_events=('cell', 'count'))
    )

    n_players = len(player_info)
    n_cells   = x_bins * y_bins
    occ_raw   = np.zeros((n_players, n_cells), dtype=np.float32)

    pid_to_row = {int(pid): i for i, pid in enumerate(player_info['player_id'])}

    counts = spatial.groupby(['player_id', 'cell']).size().reset_index(name='cnt')
    for row in counts.itertuples(index=False):
        r = pid_to_row.get(int(row.player_id))
        if r is not None and 0 <= int(row.cell) < n_cells:
            occ_raw[r, int(row.cell)] = row.cnt

    row_sums = occ_raw.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    occ_matrix = (occ_raw / row_sums).astype(np.float64)

    return player_info, occ_matrix


occ_players, occ_matrix = build_occurrence_matrix(events, player_quality, goalkeeper_ids, OMAT_X_BINS, OMAT_Y_BINS)
print(f'Jogadores no espaço espacial: {len(occ_players):,}')
print(f'Dimensão do vetor:            {occ_matrix.shape[1]}  ({OMAT_X_BINS}×{OMAT_Y_BINS})')


In [ ]:
from scipy.spatial.distance import cdist

def compute_pca_snapshot(X_in, variance_target=0.90, eps=1e-12):
    X_mean     = X_in.mean(axis=0, keepdims=True)
    X_centered = X_in - X_mean
    n = X_centered.shape[0]

    gram     = (X_centered @ X_centered.T) / max(n - 1, 1)
    eigvals, eigvecs = np.linalg.eigh(gram)
    order    = np.argsort(eigvals)[::-1]
    eigvals  = eigvals[order];  eigvecs = eigvecs[:, order]

    valid   = eigvals > eps
    eigvals = eigvals[valid];   eigvecs = eigvecs[:, valid]

    singular = np.sqrt(eigvals * max(n - 1, 1))
    cumul    = np.cumsum(eigvals / eigvals.sum())
    k_var    = int(np.searchsorted(cumul, variance_target) + 1)
    k        = int(min(max(2, k_var), eigvecs.shape[1]))

    spatial_modes = (X_centered.T @ eigvecs[:, :k]) / singular[:k]
    scores        = eigvecs[:, :k] * singular[:k]
    embedding     = scores / np.sqrt(np.maximum(eigvals[:k], eps))

    return X_mean, eigvals, embedding, spatial_modes, cumul, k


occ_X_mean, occ_eigvals, occ_embedding, occ_spatial_modes, occ_cumul, occ_k = compute_pca_snapshot(
    occ_matrix, variance_target=OCC_VARIANCE_TARGET
)

print(f'Componentes selecionados: {occ_k}')
print(f'Variância explicada:      {occ_cumul[occ_k - 1]:.4f}')
print(f'Shape do embedding:       {occ_embedding.shape}')

occ_cosine_dist = cdist(occ_embedding, occ_embedding, metric='cosine')
occ_pid_to_idx  = {int(pid): i for i, pid in enumerate(occ_players['player_id'])}


def most_similar_player(focal_pid, candidate_pids):
    fi = occ_pid_to_idx.get(int(focal_pid))
    if fi is None:
        return None
    best_dist, best_pid = np.inf, None
    for cid in candidate_pids:
        ci = occ_pid_to_idx.get(int(cid))
        if ci is None or ci == fi:
            continue
        d = float(occ_cosine_dist[fi, ci])
        if d < best_dist:
            best_dist, best_pid = d, int(cid)
    return best_pid


## **13. Determinação do Jogador Proxy por Jogo**

Para cada par `(jogador, partida)`, o proxy é determinado por uma das três regras:

| Caso | Condição | Método | Proxy |
|------|----------|--------|-------|
| 1 | Titular e jogou a partida toda (sem ser substituído) | `full_match` | — |
| 2 | Titular mas foi substituído no decorrer do jogo | `direct_sub` | Jogador que entrou em seu lugar |
| 3 | Não começou a partida (0 min ou entrou como substituto) | `spatial_similarity` | Jogador com vetor de ocorrências mais similar |


In [ ]:
def build_proxy_per_match(full_pm, player_lk):
    subs_ev = events.loc[
        events['type'].eq('Substitution') &
        events['player_id'].notna() &
        events['substitution_replacement_id'].notna(),
        ['match_id', 'team_id', 'player_id', 'substitution_replacement_id']
    ].copy()
    subs_ev['player_id']                   = subs_ev['player_id'].astype(int)
    subs_ev['substitution_replacement_id'] = subs_ev['substitution_replacement_id'].astype(int)
    subs_ev['team_id']                     = subs_ev['team_id'].astype(int)
    sub_lookup = {
        (r.match_id, r.team_id, r.player_id): r.substitution_replacement_id
        for r in subs_ev.itertuples(index=False)
    }

    name_lk = player_lk.set_index('player_id')['player'].to_dict()

    team_roster = {
        int(k): v for k, v in full_pm.groupby('team_id')['player_id'].apply(set).to_dict().items()
    }

    rows = []
    for rec in full_pm.itertuples(index=False):
        pid     = int(rec.player_id)
        mid     = rec.match_id
        tid     = int(rec.team_id)

        if rec.started and not rec.subbed_out:
            proxy_id, proxy_method = None, 'full_match'

        elif rec.started and rec.subbed_out:
            proxy_id     = sub_lookup.get((mid, tid, pid))
            proxy_method = 'direct_sub' if proxy_id is not None else 'spatial_similarity'
            if proxy_id is None:
                candidates = team_roster.get(tid, set()) - {pid}
                proxy_id   = most_similar_player(pid, candidates)

        else:
            candidates = team_roster.get(tid, set()) - {pid}
            proxy_id   = most_similar_player(pid, candidates)
            proxy_method = 'spatial_similarity' if proxy_id is not None else 'no_proxy'

        rows.append({
            'match_id':     mid,
            'team_id':      tid,
            'team':         rec.team,
            'player_id':    pid,
            'player':       rec.player,
            'minutes':      float(rec.minutes),
            'started':      rec.started,
            'subbed_out':   rec.subbed_out,
            'came_as_sub':  rec.came_as_sub,
            'proxy_id':     proxy_id,
            'proxy_player': name_lk.get(int(proxy_id)) if proxy_id is not None else None,
            'proxy_method': proxy_method,
        })

    return pd.DataFrame(rows)


player_match_proxy = build_proxy_per_match(full_player_match_min, player_lookup)

print(f'Total registros (jogador × partida): {len(player_match_proxy):,}')
print('\nDistribuição por método de proxy:')
print(player_match_proxy['proxy_method'].value_counts())


In [ ]:
player_match_proxy.to_parquet(PROCESSED_PATH / 'player_match_proxy.parquet', index=False)
print(f'Salvo em: {PROCESSED_PATH / "player_match_proxy.parquet"}')

(
    player_match_proxy[player_match_proxy['team_id'].eq(sample_team_id)]
    .sort_values(['match_id', 'proxy_method', 'minutes'], ascending=[True, True, False])
    [['team', 'match_id', 'player', 'minutes', 'started', 'subbed_out',
      'came_as_sub', 'proxy_method', 'proxy_player']]
    .head(30)
)


## **14. Configurações de Sinergia**

In [ ]:
MIN_TOGETHER_MINUTES = 300.0
MIN_REPLACEMENT_MINUTES = 180.0

## **15. Lookup de Posições e Perfis Espaciais**

Cada jogador elegível por minutagem é representado por um histograma 2D normalizado de localização de eventos. PCA compacta para `k` componentes que explicam ≥90% da variância, sem teto manual. Similaridade = cosseno entre embeddings normalizados.

In [ ]:
NON_ROLE_EVENT_TYPES = {
    'Starting XI','Half Start','Half End','Tactical Shift',
    'Substitution','Player Off','Player On','Injury Stoppage',
    'Bad Behaviour','Referee Ball-Drop',
}

def build_position_lookup(ev):
    rows = []
    for row in ev.loc[ev['type'].eq('Starting XI'), ['match_id','team_id','team','tactics']].itertuples(index=False):
        tactics = parse_tactics(row.tactics)
        for item in tactics.get('lineup', []):
            player   = item.get('player', {})
            position = item.get('position', {})
            pid = player.get('id')
            if pid is None: continue
            rows.append({'team_id':int(row.team_id),'team':row.team,'player_id':int(pid),'lineup_position':position.get('name')})
    if not rows:
        return pd.DataFrame(columns=['team_id','player_id','lineup_position'])
    positions = pd.DataFrame(rows)
    return (
        positions.groupby(['team_id','team','player_id'], as_index=False)
        .agg(lineup_position=('lineup_position', lambda x: x.dropna().astype(str).mode().iloc[0] if not x.dropna().empty else None))
    )

def build_role_profiles(ev, player_lk, pos_lk, player_qual, gk_ids):
    eligible_players = set(
        player_qual.loc[player_qual['min_total'].ge(MIN_EIGENMAP_MINUTES), 'player_id']
        .dropna()
        .astype(int)
    )

    role_ev = ev.loc[
        ev['team_id'].notna() & ev['player_id'].notna() &
        ev['location_x'].notna() & ev['location_y'].notna() &
        ~ev['type'].isin(NON_ROLE_EVENT_TYPES),
        ['team_id','team','player_id','player','type','location_x','location_y']
    ].copy()
    role_ev['team_id']   = role_ev['team_id'].astype(int)
    role_ev['player_id'] = role_ev['player_id'].astype(int)
    role_ev = role_ev[~role_ev['player_id'].isin(gk_ids)]
    role_ev = role_ev[role_ev['player_id'].isin(eligible_players)].copy()
    role_ev['location_x'] = pd.to_numeric(role_ev['location_x'], errors='coerce')
    role_ev['location_y'] = pd.to_numeric(role_ev['location_y'], errors='coerce')
    role_ev = role_ev[role_ev['location_x'].between(0,120) & role_ev['location_y'].between(0,80)].copy()

    x_edges = np.linspace(0, 120, OMAT_X_BINS + 1)
    y_edges = np.linspace(0, 80,  OMAT_Y_BINS + 1)

    rows, vectors = [], []
    for (team_id, player_id), grp in role_ev.groupby(['team_id','player_id'], sort=True):
        hist, _, _ = np.histogram2d(grp['location_y'].to_numpy(float), grp['location_x'].to_numpy(float), bins=[y_edges, x_edges])
        total = hist.sum()
        if total <= 0: continue
        density = (hist / total).astype(float)
        vectors.append(density.reshape(-1))
        rows.append({'team_id':int(team_id),'player_id':int(player_id),'role_events':int(len(grp)),
                     'mean_x':float(grp['location_x'].mean()),'mean_y':float(grp['location_y'].mean()),
                     'defensive_third_share':float((grp['location_x']<40).mean()),
                     'attacking_third_share':float((grp['location_x']>=80).mean())})

    profiles  = pd.DataFrame(rows)
    x_matrix  = np.vstack(vectors)
    x_ctr     = x_matrix - x_matrix.mean(axis=0, keepdims=True)
    _, sv, vt = np.linalg.svd(x_ctr, full_matrices=False)
    eigvals   = (sv**2) / max(len(x_matrix)-1, 1)
    cumulative= np.cumsum(eigvals / eigvals.sum())
    k = int(min(max(2, int(np.searchsorted(cumulative, OCC_VARIANCE_TARGET)+1)), vt.shape[0]))
    scores = x_ctr @ vt[:k].T
    scores = (scores - scores.mean(axis=0, keepdims=True)) / (scores.std(axis=0, keepdims=True) + 1e-12)
    embeddings = scores / np.maximum(np.linalg.norm(scores, axis=1, keepdims=True), 1e-12)

    profiles = pd.concat([
        profiles.reset_index(drop=True),
        pd.DataFrame({f'role_pc_{i+1}' : scores[:,i]     for i in range(k)}),
        pd.DataFrame({f'role_emb_{i+1}': embeddings[:,i] for i in range(k)}),
        pd.DataFrame({f'role_bin_{i:02d}': x_matrix[:,i] for i in range(x_matrix.shape[1])}),
    ], axis=1)
    profiles = profiles.merge(player_lk, on=['team_id','player_id'], how='left')
    profiles = profiles.merge(pos_lk[['team_id','player_id','lineup_position']], on=['team_id','player_id'], how='left')
    profiles['role_components']         = k
    profiles['role_explained_variance'] = float(cumulative[k-1])
    profiles.to_parquet(PROCESSED_PATH / 'player_role_profiles.parquet', index=False)

    emb_cols = [f'role_emb_{i+1}' for i in range(k)]
    sim_rows = []
    for team_id, tp in profiles.groupby('team_id', sort=False):
        ids = tp['player_id'].astype(int).to_numpy()
        emb = tp[emb_cols].to_numpy(float)
        if len(ids) < 2: continue
        sim = emb @ emb.T
        for i, t in enumerate(ids):
            for j, c in enumerate(ids):
                if i == j: continue
                sim_rows.append({'team_id':int(team_id),'target_player_id':int(t),'candidate_player_id':int(c),'role_similarity':float(sim[i,j])})
    similarities = pd.DataFrame(sim_rows)
    similarities.to_parquet(PROCESSED_PATH / 'player_role_similarity.parquet', index=False)
    return profiles, similarities

position_lookup            = build_position_lookup(events)
role_profiles, similarities = build_role_profiles(events, player_lookup, position_lookup, player_quality, goalkeeper_ids)
print(f'Perfis: {len(role_profiles):,}  |  Pares de similaridade: {len(similarities):,}')

## **16. Seleção de Proxy e Acumulação de Amostras**

Para cada segmento com jogador ausente, o proxy é escolhido por:
1. Ocupante atual do mesmo slot de substituição.
2. Titular do slot com maior compatibilidade posicional + similaridade de eigenmap.

In [ ]:
def _norm_pos(p):
    return '' if (p is None or (isinstance(p, float) and np.isnan(p))) else str(p).strip().lower()

def _pos_side(p):
    t = _norm_pos(p)
    if t.startswith('left')  or ' left '  in f' {t} ': return 'left'
    if t.startswith('right') or ' right ' in f' {t} ': return 'right'
    return None

def _pos_role(p):
    t = _norm_pos(p)
    if not t: return 'unknown'
    if 'goalkeeper'         in t: return 'goalkeeper'
    if 'center back'        in t or 'centre back' in t: return 'center_back'
    if 'back'               in t: return 'fullback'
    if 'defensive midfield' in t: return 'defensive_midfield'
    if 'attacking midfield' in t: return 'attacking_midfield'
    if 'midfield'           in t: return 'central_midfield'
    if 'wing'               in t: return 'winger'
    if 'forward' in t or 'striker' in t: return 'forward'
    return 'unknown'

def _pos_compat(target, candidate):
    tr, cr = _pos_role(target), _pos_role(candidate)
    if tr in {'goalkeeper','unknown'} or cr in {'goalkeeper','unknown'}: return 0
    ts, cs = _pos_side(target), _pos_side(candidate)
    tn, cn = _norm_pos(target), _norm_pos(candidate)
    if tn == cn: return 6
    table = {
        ('winger','winger'):(5 if ts and ts==cs else 4),('winger','forward'):3,('winger','attacking_midfield'):2,
        ('forward','forward'):5,('forward','winger'):4,('forward','attacking_midfield'):3,
        ('attacking_midfield','attacking_midfield'):5,('attacking_midfield','central_midfield'):4,
        ('attacking_midfield','winger'):3,('attacking_midfield','forward'):3,('attacking_midfield','defensive_midfield'):2,
        ('central_midfield','central_midfield'):5,('central_midfield','defensive_midfield'):4,('central_midfield','attacking_midfield'):4,
        ('defensive_midfield','defensive_midfield'):5,('defensive_midfield','central_midfield'):4,('defensive_midfield','center_back'):2,
        ('fullback','fullback'):(5 if ts and ts==cs else 3),('fullback','center_back'):1,
        ('center_back','center_back'):5,('center_back','fullback'):2,('center_back','defensive_midfield'):2,
    }
    return table.get((tr,cr), 0)

def _sim_or_zero(sim_lk, team_id, target, candidate):
    v = sim_lk.get((int(team_id), int(target), int(candidate)))
    return float(v) if v is not None and np.isfinite(v) else 0.0

def choose_proxy(seg, team_id, focal_id, absent_id, active, sim_lk, pos_lk_dict, gk_ids):
    cur_by_slot = {int(s): int(p) for s,p in seg.slot_players}
    p2s         = {int(p): int(s) for p,s in seg.match_player_slots}
    starters    = {int(s): int(p) for s,p,_ in seg.starter_slots}
    starter_pos = {int(s): pos    for s,_,pos in seg.starter_slots}

    if int(absent_id) in p2s:
        sid = p2s[int(absent_id)]
        prx = cur_by_slot.get(sid)
        if prx and prx != int(focal_id) and prx in active and prx not in gk_ids:
            return int(prx), int(absent_id), 'same_substitution_slot', _sim_or_zero(sim_lk, team_id, absent_id, prx)
        return None, None, None, np.nan

    target_pos = pos_lk_dict.get((int(team_id), int(absent_id)))
    best = None
    for sid, stid in starters.items():
        prx = cur_by_slot.get(int(sid))
        if not prx or stid in gk_ids or prx in gk_ids: continue
        if stid == int(focal_id) or prx == int(focal_id) or prx not in active: continue
        compat = _pos_compat(target_pos, starter_pos.get(int(sid)))
        if compat <= 0: continue
        sim = sim_lk.get((int(team_id), int(absent_id), stid))
        if sim is None or not np.isfinite(sim): continue
        cand = (int(compat), float(sim), int(sid), stid, int(prx))
        if best is None or cand[:2] > best[:2]: best = cand

    if best is None: return None, None, None, np.nan
    _, sim, _, anchor, prx = best
    return int(prx), int(anchor), 'starter_similarity_slot', float(sim)

In [ ]:
def build_player_windows(pm, gk_ids):
    w = (
        pm[~pm['player_id'].isin(gk_ids)]
        .assign(match_date=lambda d: pd.to_datetime(d['match_date']))
        .groupby(['season','team_id','player_id'], as_index=False)
        .agg(first_date=('match_date','min'), last_date=('match_date','max'), min_total=('minutes','sum'))
    )
    w['team_id']   = w['team_id'].astype(int)
    w['player_id'] = w['player_id'].astype(int)
    return w

def aggregate_role_adjusted_samples(segs, vaep_act, windows, role_prof, sims, pos_lk, gk_ids):
    FIELDS = ['minutes_together','focal_vaep_together','partner_vaep_together','duo_vaep_together',
              'duo_pos_vaep_together','focal_actions_together','partner_actions_together','duo_actions_together',
              'minutes_proxy','focal_vaep_proxy','proxy_vaep','proxy_pos_vaep','proxy_duo_vaep',
              'proxy_duo_pos_vaep','focal_actions_proxy','proxy_actions','proxy_duo_actions',
              'proxy_similarity_minutes','proxy_segments','same_slot_proxy_minutes','starter_similarity_proxy_minutes']
    samp_dict = defaultdict(lambda: {f: 0.0 for f in FIELDS})
    prx_min   = defaultdict(float)
    prx_sim   = defaultdict(float)
    sim_lk = {(int(r.team_id), int(r.target_player_id), int(r.candidate_player_id)): float(r.role_similarity)
               for r in sims.itertuples(index=False)}
    pos_lk_dict = {(int(r.team_id), int(r.player_id)): r.lineup_position
                    for r in pos_lk[['team_id','player_id','lineup_position']].dropna(subset=['team_id','player_id']).itertuples(index=False)}
    profiled = {(int(r.team_id), int(r.player_id)) for r in role_prof[['team_id','player_id']].itertuples(index=False)}

    action_base = (
        vaep_act.loc[vaep_act['player_id'].notna(), ['match_id','team_id','event_minute','player_id','vaep']].copy()
        .assign(team_id=lambda d: d['team_id'].astype(int), player_id=lambda d: d['player_id'].astype(int))
    )
    action_base = action_base[~action_base['player_id'].isin(gk_ids)]
    action_groups = {k: g.sort_values('event_minute')[['event_minute','player_id','vaep']]
                     for k, g in action_base.groupby(['match_id','team_id'], sort=False)}
    window_groups = {k: g[['player_id','first_date','last_date']].copy()
                     for k, g in windows.groupby(['season','team_id'], sort=False)}

    for seg in segs.itertuples(index=False):
        team_id = int(seg.team_id)
        active_all      = {int(p) for p in seg.active if int(p) not in gk_ids}
        active_profiled = {p for p in active_all if (team_id, p) in profiled}
        if len(active_profiled) < 1: continue

        wg = window_groups.get((seg.season, team_id))
        if wg is None or wg.empty: continue

        mdate     = pd.to_datetime(seg.match_date)
        available = set(wg.loc[wg['first_date'].le(mdate) & wg['last_date'].ge(mdate), 'player_id'].astype(int)) | active_profiled
        available = {int(p) for p in available if int(p) not in gk_ids and (team_id, int(p)) in profiled}
        absent    = available - active_profiled

        ag = action_groups.get((seg.match_id, team_id))
        if ag is None:
            seg_actions = pd.DataFrame(columns=['player_id','vaep'])
        else:
            mask = ag['event_minute'].ge(seg.start_minute) & ag['event_minute'].lt(seg.end_minute)
            seg_actions = ag.loc[mask][lambda d: d['player_id'].isin(active_all)]

        vaep_map    = seg_actions.groupby('player_id')['vaep'].sum().to_dict()
        pos_map     = seg_actions.assign(pv=seg_actions['vaep'].clip(lower=0)).groupby('player_id')['pv'].sum().to_dict()
        actions_map = seg_actions['player_id'].value_counts().to_dict()

        for focal_id in sorted(active_profiled):
            fv = float(vaep_map.get(focal_id, 0.0))
            fpv= float(pos_map.get(focal_id, 0.0))
            fa = int(actions_map.get(focal_id, 0))

            for partner_id in sorted(active_profiled):
                if partner_id == focal_id: continue
                key = (team_id, seg.season, int(focal_id), int(partner_id), int(seg.match_id))
                r   = samp_dict[key]
                pv  = float(vaep_map.get(partner_id, 0.0))
                ppv = float(pos_map.get(partner_id, 0.0))
                pa  = int(actions_map.get(partner_id, 0))
                r['minutes_together']      += float(seg.duration)
                r['focal_vaep_together']   += fv
                r['partner_vaep_together'] += pv
                r['duo_vaep_together']     += fv + pv
                r['duo_pos_vaep_together'] += fpv + ppv
                r['focal_actions_together'] += fa
                r['partner_actions_together'] += pa
                r['duo_actions_together']  += fa + pa

            for absent_id in absent:
                prx_id, anchor_id, method, sim = choose_proxy(seg, team_id, focal_id, absent_id, active_all, sim_lk, pos_lk_dict, gk_ids)
                if prx_id is None or not np.isfinite(sim): continue
                key  = (team_id, seg.season, int(focal_id), int(absent_id), int(seg.match_id))
                r    = samp_dict[key]
                prxv = float(vaep_map.get(prx_id, 0.0))
                prxpv= float(pos_map.get(prx_id, 0.0))
                prxa = int(actions_map.get(prx_id, 0))
                r['minutes_proxy']             += float(seg.duration)
                r['focal_vaep_proxy']          += fv
                r['proxy_vaep']                += prxv
                r['proxy_pos_vaep']            += prxpv
                r['proxy_duo_vaep']            += fv + prxv
                r['proxy_duo_pos_vaep']        += fpv + prxpv
                r['focal_actions_proxy']       += fa
                r['proxy_actions']             += prxa
                r['proxy_duo_actions']         += fa + prxa
                r['proxy_similarity_minutes']  += float(sim) * float(seg.duration)
                r['proxy_segments']            += 1
                if method == 'same_substitution_slot':    r['same_slot_proxy_minutes']           += float(seg.duration)
                elif method == 'starter_similarity_slot': r['starter_similarity_proxy_minutes']  += float(seg.duration)
                ck = (team_id, int(focal_id), int(absent_id), int(prx_id), int(anchor_id), method)
                prx_min[ck] += float(seg.duration)
                prx_sim[ck] += float(sim) * float(seg.duration)

    sample_rows = [{'team_id':k[0],'season':k[1],'focal_player_id':k[2],'partner_player_id':k[3],'match_id':k[4], **v}
                   for k, v in samp_dict.items()]
    samples_df = pd.DataFrame(sample_rows)

    choice_rows = [{'team_id':k[0],'focal_player_id':k[1],'partner_player_id':k[2],'proxy_player_id':k[3],
                    'proxy_anchor_player_id':k[4],'proxy_method':k[5],'proxy_minutes':m,
                    'mean_role_similarity':prx_sim[k]/m if m>0 else np.nan}
                   for k, m in prx_min.items()]
    proxy_choices = pd.DataFrame(choice_rows)

    return samples_df, proxy_choices

player_windows = build_player_windows(player_match, goalkeeper_ids)
samples, proxy_choices = aggregate_role_adjusted_samples(
    segments, vaep_actions, player_windows, role_profiles, similarities, position_lookup, goalkeeper_ids
)
samples.to_parquet(PROCESSED_PATH / 'role_adjusted_context_samples.parquet', index=False)
proxy_choices.to_parquet(PROCESSED_PATH / 'role_adjusted_proxy_choices.parquet', index=False)
print(f'Amostras: {len(samples):,}  |  Escolhas de proxy: {len(proxy_choices):,}')

## **17. Sinergia Direcionada (Role-Adjusted)**

`synergy_score(A→B) = VAEP90(A+B juntos) − VAEP90(A+proxy_de_B)`

In [ ]:
def add_role_adjusted_metrics(df):
    o = df.copy()
    safe = lambda x: x.replace(0, np.nan)
    o['duo_vaep90_together']    = 90 * o['duo_vaep_together']    / safe(o['minutes_together'])
    o['focal_vaep90_together']  = 90 * o['focal_vaep_together']  / safe(o['minutes_together'])
    o['partner_vaep90_together']= 90 * o['partner_vaep_together']/ safe(o['minutes_together'])
    o['proxy_duo_vaep90']       = 90 * o['proxy_duo_vaep']       / safe(o['minutes_proxy'])
    o['focal_vaep90_proxy']     = 90 * o['focal_vaep_proxy']     / safe(o['minutes_proxy'])
    o['proxy_vaep90']           = 90 * o['proxy_vaep']           / safe(o['minutes_proxy'])
    o['mean_proxy_similarity']  = o['proxy_similarity_minutes']  / safe(o['minutes_proxy'])
    o['same_slot_proxy_share']  = o['same_slot_proxy_minutes']   / safe(o['minutes_proxy'])
    o['starter_similarity_proxy_share'] = o['starter_similarity_proxy_minutes'] / safe(o['minutes_proxy'])
    o['duo_impact_p90']         = o['duo_vaep90_together'] - o['proxy_duo_vaep90']
    o['focal_vaep90_with_partner'] = o['focal_vaep90_together']
    o['focal_vaep90_with_proxy']   = o['focal_vaep90_proxy']
    o['focal_impact_p90']       = o['focal_vaep90_with_partner'] - o['focal_vaep90_with_proxy']
    o['synergy_score']          = o['duo_impact_p90']
    return o

def summarize_directed_synergy(samps, prx_ch, player_lk, player_qual):
    agg = samps.groupby(['team_id','focal_player_id','partner_player_id'], as_index=False).agg(
        minutes_together=('minutes_together','sum'), focal_vaep_together=('focal_vaep_together','sum'),
        partner_vaep_together=('partner_vaep_together','sum'), duo_vaep_together=('duo_vaep_together','sum'),
        focal_actions_together=('focal_actions_together','sum'), partner_actions_together=('partner_actions_together','sum'),
        duo_actions_together=('duo_actions_together','sum'), minutes_proxy=('minutes_proxy','sum'),
        focal_vaep_proxy=('focal_vaep_proxy','sum'), proxy_vaep=('proxy_vaep','sum'),
        proxy_duo_vaep=('proxy_duo_vaep','sum'), focal_actions_proxy=('focal_actions_proxy','sum'),
        proxy_actions=('proxy_actions','sum'), proxy_duo_actions=('proxy_duo_actions','sum'),
        proxy_similarity_minutes=('proxy_similarity_minutes','sum'), proxy_segments=('proxy_segments','sum'),
        same_slot_proxy_minutes=('same_slot_proxy_minutes','sum'),
        starter_similarity_proxy_minutes=('starter_similarity_proxy_minutes','sum'),
        matches_together=('minutes_together', lambda x: int((x>0).sum())),
        matches_proxy=('minutes_proxy', lambda x: int((x>0).sum())),
    )
    directed = add_role_adjusted_metrics(agg)
    directed = directed.loc[
        directed['minutes_together'].ge(MIN_TOGETHER_MINUTES) &
        directed['minutes_proxy'].ge(MIN_REPLACEMENT_MINUTES) &
        directed['duo_vaep90_together'].notna() &
        directed['proxy_duo_vaep90'].notna()
    ].copy()

    top_choice = (
        prx_ch.sort_values('proxy_minutes', ascending=False)
        .drop_duplicates(['team_id','focal_player_id','partner_player_id'])
        .rename(columns={'proxy_player_id':'main_proxy_player_id','proxy_anchor_player_id':'main_proxy_anchor_player_id',
                         'proxy_method':'main_proxy_method','proxy_minutes':'main_proxy_minutes',
                         'mean_role_similarity':'main_proxy_similarity'})
    )
    directed = directed.merge(
        top_choice[['team_id','focal_player_id','partner_player_id','main_proxy_player_id',
                    'main_proxy_anchor_player_id','main_proxy_method','main_proxy_minutes','main_proxy_similarity']],
        on=['team_id','focal_player_id','partner_player_id'], how='left'
    )

    quality = player_qual.copy()
    if 'vaep_p90' not in quality.columns:
        quality['vaep_p90'] = 90 * quality['vaep_total'] / quality['min_total'].replace(0, np.nan)
    q_map = quality.set_index('player_id')['vaep_p90'].to_dict()
    directed['focal_vaep90_total']      = directed['focal_player_id'].map(q_map)
    directed['partner_vaep90_total']    = directed['partner_player_id'].map(q_map)
    directed['main_proxy_vaep90_total'] = directed['main_proxy_player_id'].map(q_map)

    fn = player_lk.rename(columns={'player_id':'focal_player_id','player':'focal_player','team':'team'})[['team_id','focal_player_id','focal_player','team']]
    pn = player_lk.rename(columns={'player_id':'partner_player_id','player':'partner_player'})[['team_id','partner_player_id','partner_player']]
    xn = player_lk.rename(columns={'player_id':'main_proxy_player_id','player':'main_proxy_player'})[['team_id','main_proxy_player_id','main_proxy_player']]
    directed = directed.merge(fn, on=['team_id','focal_player_id'], how='left')
    directed = directed.merge(pn, on=['team_id','partner_player_id'], how='left')
    directed = directed.merge(xn, on=['team_id','main_proxy_player_id'], how='left')
    return directed.sort_values('synergy_score', ascending=False).reset_index(drop=True)

directed = summarize_directed_synergy(samples, proxy_choices, player_lookup, player_quality)
directed.to_parquet(PROCESSED_PATH / 'directed_synergy_all.parquet', index=False)
print(f'Relações direcionadas elegíveis: {len(directed):,}')

In [ ]:
MIN_TOGETHER_DISPLAY = 300
MIN_PROXY_DISPLAY    = 180

impact = directed.loc[
    directed['minutes_together'].ge(MIN_TOGETHER_DISPLAY) &
    directed['minutes_proxy'].ge(MIN_PROXY_DISPLAY)
].copy()
impact['absolute_impact_p90'] = impact['duo_impact_p90']

display_cols = ['team','focal_player','partner_player','main_proxy_player','absolute_impact_p90',
                'duo_vaep90_together','proxy_duo_vaep90','focal_vaep90_with_partner',
                'focal_vaep90_with_proxy','focal_impact_p90',
                'minutes_together','minutes_proxy','main_proxy_method']
display_cols = [c for c in display_cols if c in impact.columns]
impact[display_cols].sort_values('absolute_impact_p90', ascending=False).head(20).round(3)


In [ ]:
barcelona_impact = impact.loc[impact['team'].eq('Barcelona')].copy()
barcelona_impact['proxy_player'] = barcelona_impact['main_proxy_player']
barcelona_display_cols = ['focal_player','proxy_player','absolute_impact_p90','focal_impact_p90','minutes_together','minutes_proxy', 'main_proxy_method']

def show_barcelona_partner(partner_pattern):
    partner_mask = barcelona_impact['partner_player'].fillna('').str.contains(partner_pattern, case=False, regex=True)
    return (
        barcelona_impact.loc[partner_mask, barcelona_display_cols]
        .sort_values('absolute_impact_p90', ascending=False)
        .round(3)
        .reset_index(drop=True)
    )


In [ ]:
barcelona_messi_partner = show_barcelona_partner('Messi|Lionel')
barcelona_messi_partner

In [ ]:
barcelona_messi_partner = show_barcelona_partner('Neymar')
barcelona_messi_partner

In [ ]:
barcelona_munir_partner = show_barcelona_partner('Munir|Haddadi')
barcelona_munir_partner